# Machine learning pipelines
Predict Titanic survival twice: first with every preprocessing step done by hand, then with one scikit-learn `Pipeline`.
Then look inside the pipeline, cross-validate it, tune it with a grid search, save it, and use it on a new passenger.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.base import clone

# folder for the saved models
Path("models").mkdir(exist_ok=True)

## 1. The data

In [ ]:
# Titanic passengers (Kaggle's train.csv): 891 rows
df = pd.read_csv("data/titanic_train.csv")
df.head()

In [ ]:
# drop the four columns we will not use
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
df.head()

In [ ]:
# missing values per column: Age has 177, Embarked has 2
df.isnull().sum()

In [ ]:
# inputs = every column except Survived; split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["Survived"]),
    df["Survived"],
    test_size=0.2,
    random_state=42)
X_train.shape, X_test.shape

In [ ]:
# the first rows of the training inputs
X_train.head()

## 2. Without a pipeline: every step by hand

In [ ]:
# step 1: fill missing values, one imputer per column
si_age = SimpleImputer()                              # mean
si_embarked = SimpleImputer(strategy="most_frequent") # mode

X_train_age = si_age.fit_transform(X_train[["Age"]])
X_train_embarked = si_embarked.fit_transform(X_train[["Embarked"]])

X_test_age = si_age.transform(X_test[["Age"]])
X_test_embarked = si_embarked.transform(X_test[["Embarked"]])

In [ ]:
# step 2: one-hot encode Sex and Embarked, one encoder per column
ohe_sex = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe_embarked = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

X_train_sex = ohe_sex.fit_transform(X_train[["Sex"]])
# Embarked: encode the imputed array, not the original column
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

X_test_sex = ohe_sex.transform(X_test[["Sex"]])
X_test_embarked = ohe_embarked.transform(X_test_embarked)
X_train_embarked[:3]

In [ ]:
# step 3: the untouched columns, then join everything side by side
X_train_rem = X_train.drop(columns=["Sex", "Age", "Embarked"])
X_test_rem = X_test.drop(columns=["Sex", "Age", "Embarked"])

X_train_transformed = np.concatenate(
    (X_train_rem, X_train_age, X_train_sex, X_train_embarked), axis=1)
X_test_transformed = np.concatenate(
    (X_test_rem, X_test_age, X_test_sex, X_test_embarked), axis=1)
X_train_transformed.shape, X_test_transformed.shape

In [ ]:
# step 4: train a decision tree and test it
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train_transformed, y_train)
y_pred = clf.predict(X_test_transformed)
accuracy_score(y_test, y_pred)

In [ ]:
# step 5: save the fitted objects the website will need
pickle.dump(ohe_sex, open("models/ohe_sex.pkl", "wb"))
pickle.dump(ohe_embarked, open("models/ohe_embarked.pkl", "wb"))
pickle.dump(clf, open("models/clf.pkl", "wb"))

### Predicting for a new passenger, without a pipeline

In [ ]:
# (this is what the website's code would do)
ohe_sex = pickle.load(open("models/ohe_sex.pkl", "rb"))
ohe_embarked = pickle.load(open("models/ohe_embarked.pkl", "rb"))
clf = pickle.load(open("models/clf.pkl", "rb"))

# one new passenger: Pclass, Sex, Age, SibSp, Parch, Fare, Embarked
new = pd.DataFrame([[2, "male", 31.0, 0, 0, 10.5, "S"]],
                   columns=X_train.columns)
new

In [ ]:
# repeat every training step, in the same order
new_sex = ohe_sex.transform(new[["Sex"]])
# ohe_embarked was fitted on a bare array, so give it one too
new_embarked = ohe_embarked.transform(new[["Embarked"]].values)
new_age = new[["Age"]].values
new_rem = new[["Pclass", "SibSp", "Parch", "Fare"]].values

# same order as in training: rest, age, sex, embarked
new_transformed = np.concatenate(
    (new_rem, new_age, new_sex, new_embarked), axis=1)
new_transformed.shape

In [ ]:
# 1 = survives, 0 = does not
clf.predict(new_transformed)

## 3. With a pipeline

In [ ]:
# step 1: imputation (positions: 2 = Age, 6 = Embarked)
trf1 = ColumnTransformer([
    ("impute_age", SimpleImputer(), [2]),
    ("impute_embarked", SimpleImputer(strategy="most_frequent"), [6]),
], remainder="passthrough")

In [ ]:
# what trf1 outputs: the transformed columns come first,
# so the order is now Age, Embarked, Pclass, Sex, SibSp, Parch, Fare
trf1.fit_transform(X_train)[:3]

In [ ]:
# step 2: one-hot encoding. Positions refer to trf1's OUTPUT:
# 1 = Embarked, 3 = Sex
trf2 = ColumnTransformer([
    ("ohe_sex_embarked",
     OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
     [1, 3]),
], remainder="passthrough")

In [ ]:
# step 3: scale all 10 columns to the range 0 to 1
trf3 = ColumnTransformer([
    ("scale", MinMaxScaler(), slice(0, 10)),
])

In [ ]:
# step 4: keep the 8 best columns according to the chi-squared test
trf4 = SelectKBest(score_func=chi2, k=8)

In [ ]:
# step 5: the model
trf5 = DecisionTreeClassifier(random_state=42)

In [ ]:
# join the five steps: a list of (name, object) tuples
pipe = Pipeline([
    ("trf1", trf1),
    ("trf2", trf2),
    ("trf3", trf3),
    ("trf4", trf4),
    ("trf5", trf5),
])

In [ ]:
# the same pipeline without names: make_pipeline names the steps itself
make_pipeline(trf1, trf2, trf3, trf4, trf5).named_steps.keys()

In [ ]:
# train: X_train flows through all five steps
pipe.fit(X_train, y_train)

In [ ]:
# the five fitted steps, by name
pipe.named_steps

In [ ]:
# predict: X_test flows through the same fitted steps
y_pred = pipe.predict(X_test)
accuracy_score(y_test, y_pred)

### The column-order trap

In [ ]:
# positions [1, 6] for Sex and Embarked are the ORIGINAL positions.
# After trf1, position 6 is Fare: it gets one-hot encoded by mistake.
wrong = ColumnTransformer([
    ("ohe_sex_embarked",
     OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
     [1, 6]),
], remainder="passthrough")
# clone = an unfitted copy, so our fitted pipe stays untouched
wrong_pipe = clone(pipe).set_params(trf2=wrong)
wrong_pipe.fit(X_train, y_train)
# columns after the wrong step 2: one per distinct fare
print(wrong_pipe[:2].transform(X_train).shape)
accuracy_score(y_test, wrong_pipe.predict(X_test))

### A pipeline without a model

In [ ]:
# only the preprocessing steps: use fit_transform, not fit + predict
# (an unfitted copy of the first three steps)
prep = clone(pipe[:3])
prep.fit_transform(X_train)[:2].round(2)

## 4. Looking inside the pipeline

In [ ]:
# the pipeline drawn as a diagram (the default display in Jupyter)
pipe

In [ ]:
# the mean age learned by the first imputer
pipe.named_steps["trf1"].transformers_[0][1].statistics_

In [ ]:
# the most frequent port learned by the second imputer
pipe.named_steps["trf1"].transformers_[1][1].statistics_

In [ ]:
# which of the 10 columns the feature selector kept
pipe.named_steps["trf4"].get_support()

## 5. Cross-validation with a pipeline

In [ ]:
# 5 folds: every step is refitted on each training fold
cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy").mean()

## 6. Grid search with a pipeline

In [ ]:
# step name, two underscores, parameter name
params = {"trf5__max_depth": [1, 2, 3, 4, 5, None]}
grid = GridSearchCV(pipe, params, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)
grid.best_score_, grid.best_params_

In [ ]:
# the best pipeline, refitted on all training data, on the test set
accuracy_score(y_test, grid.predict(X_test))

## 7. Exporting the pipeline

In [ ]:
# one file holds every fitted step
pickle.dump(pipe, open("models/pipe.pkl", "wb"))

### Predicting for a new passenger, with a pipeline

In [ ]:
# (this is all the website's code needs)
pipe = pickle.load(open("models/pipe.pkl", "rb"))
new = pd.DataFrame([[2, "male", 31.0, 0, 0, 10.5, "S"]],
                   columns=X_train.columns)
pipe.predict(new)

In [ ]:
# the new passenger after each fitted step
for i in range(1, 5):
    print(pipe.steps[i - 1][0], pipe[:i].transform(new))

In [ ]:
# joblib does the same job as pickle, in one call each way
import joblib
joblib.dump(pipe, "models/pipe.joblib")
joblib.load("models/pipe.joblib").predict(new)

## 8. Extra: column names instead of positions

In [ ]:
# set_output(transform="pandas") passes DataFrames between the steps,
# so every step can pick its columns by name
pipe_named = Pipeline([
    ("trf1", ColumnTransformer([
        ("impute_age", SimpleImputer(), ["Age"]),
        ("impute_embarked", SimpleImputer(strategy="most_frequent"),
         ["Embarked"]),
    ], remainder="passthrough", verbose_feature_names_out=False)),
    ("trf2", ColumnTransformer([
        ("ohe_sex_embarked",
         OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
         ["Sex", "Embarked"]),
    ], remainder="passthrough", verbose_feature_names_out=False)),
    ("trf3", MinMaxScaler()),
    ("trf4", SelectKBest(score_func=chi2, k=8)),
    ("trf5", DecisionTreeClassifier(random_state=42)),
]).set_output(transform="pandas")

pipe_named.fit(X_train, y_train)
accuracy_score(y_test, pipe_named.predict(X_test))

In [ ]:
# the 8 columns that reach the model, by name
pipe_named[:-1].get_feature_names_out()